# Data Visualizations

In [ ]:
# import packages 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager
import matplotlib.dates as mdates
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from matplotlib.ticker import FuncFormatter

In [ ]:
# colors 

'#f5f0e8'   # body
'#1a1a1a'   # background
'#06395b'
'#0c629b'
"#458dbd"
"#adddfc"
'#1a5c3a'
"#22794c"
"#67a384"
"#b4d9c6"
"#c48b22"
"#e3b96b"
"#f7dfb3"
"#b84141"

color1 = ['#06395b', "#458dbd", "#adddfc", 'rgba(6, 57, 91, 0.3)', 'rgba(69, 141, 189, 0.3)' ]
color2 = ['#1a5c3a', "#4f916f", "#b4d9c6", 'rgba(79, 145, 111, 0.3)', 'rgba(180, 217, 198, 0.3)' ]
color3 = ["#c48b22", "#e3b96b", "#f7dfb3", 'rgba(196, 139, 34, 0.3)', 'rgba(227, 185, 107, 0.3)' ]

In [3]:
# theme 
sns.set_theme(
    style="whitegrid",
    rc={
        "figure.facecolor": "none",
        "axes.facecolor": "none",
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "figure.titlesize": 20
    }
)

font_manager.fontManager.addfont("PlayfairDisplay-VariableFont_wght.ttf")

## Overall Percent Spending used on Food

Food personal consumption expenditures / total personal consumption expenditures over time = what percent of spending is used for food over time in general


In [6]:
# import data

food_pce = pd.read_csv("../data/total/DFXARC1M027SBEA.csv", low_memory=False)
pce = pd.read_csv("../data/total/PCE.csv", low_memory=False)

In [23]:
# clean data 

# filter to 1960 and later 
food_pce_clean = food_pce.loc[food_pce['observation_date']>='1960-01-01']
pce_clean = pce.loc[pce['observation_date']>='1960-01-01']

# rename 
food_pce_clean = food_pce_clean.rename(columns={'DFXARC1M027SBEA':'food_pce'})
pce_clean = pce_clean.rename(columns={'PCE':'pce'})

# merge 
percent_food_expend = food_pce_clean.merge(pce_clean, how='outer')

# divide to get percentage
percent_food_expend['percent'] = (percent_food_expend['food_pce']/percent_food_expend['pce']) *100

In [91]:
# line chart of food as a percent of personal consumption expenditures

# data 
y = percent_food_expend["percent"]
x = percent_food_expend["observation_date"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name="",
        line=dict(
            color='#67a384',
            width=4
        ),
        fill='tozeroy',
        fillcolor='rgba(103, 163, 132, 0.18)',
        hovertemplate=
        "Food Percentage: %{y:.1f}%<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food Expenditure as a Share of Total Personal Consumption Expenditures",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=10
    ),

    yaxis=dict(
        title="Percentage",
        range=[0, 100],
        ticksuffix="%",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    )
)

# endpoint annotation
fig.add_annotation(
    x=x.iloc[-1],
    y=y.iloc[-1],
    text=f"{y.iloc[-1]:.1f}%",
    showarrow=False,
    xshift=20,
    font=dict(
        color="#f7dfb3",
        size=14
    )
)

fig.show()

In [92]:
# save fig 
fig.write_html(f"../website/charts/percent_food_expenditure.html", include_plotlyjs="cdn", full_html=False)

## Food CPI vs General CPI 

Food cpi vs general cpi  = whether food prices have inflated faster or slower than everything else in the economy over time / is food getting more expensive relative to other things Americans buy?

In [121]:
# import data 

food_cpi = pd.read_csv("../data/total/CPIUFDSL.csv", low_memory=False)
general_cpi = pd.read_csv("../data/total/CPIAUCSL.csv", low_memory=False)

In [122]:
# clean data 

# filter to 1960 and later 
food_cpi_clean = food_cpi.loc[food_cpi['observation_date']>='1960-01-01']
general_cpi_clean = general_cpi.loc[general_cpi['observation_date']>='1960-01-01']

# rename 
food_cpi_clean = food_cpi_clean.rename(columns={'CPIUFDSL':'food_cpi'})
general_cpi_clean = general_cpi_clean.rename(columns={'CPIAUCSL':'total_cpi'})

# merge 
both_cpi = food_cpi_clean.merge(general_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_food = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_cpi"].values
base_all = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "total_cpi"].values

both_cpi['food_cpi_reindex'] = (both_cpi['food_cpi'] / base_food) * 100
both_cpi['total_cpi_reindex'] = (both_cpi['total_cpi'] / base_all) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [132]:
# line chart of food as a percent of personal consumption expenditures

# data 
x = both_cpi["observation_date"]
food_cpi = both_cpi["food_cpi_reindex"]
total_cpi = both_cpi["total_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        hovertemplate=
        "All Items CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_cpi,
        mode="lines",
        name="Food",
        line=dict(
            color='#67a384',
            width=4
        ),
        hovertemplate=
        "Food CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food vs. Overall Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False
    )
)

fig.show()

In [133]:
# save fig 
fig.write_html(f"../website/charts/food_vs_overall_inflation.html", include_plotlyjs="cdn", full_html=False)

## Food at Home vs Food Away from Home

Food at home vs food away from home CPIs over time = whether eating at home vs eating out was cheaper at certain times 

In [135]:
# import data 

food_home_cpi = pd.read_csv("../data/categories/CUUR0000SAF11.csv", low_memory=False)
food_away_cpi = pd.read_csv("../data/categories/CUUR0000SEFV.csv", low_memory=False)

In [136]:
# clean data 

# filter to 1960 and later 
food_home_cpi_clean = food_home_cpi.loc[food_home_cpi['observation_date']>='1960-01-01']
food_away_cpi_clean = food_away_cpi.loc[food_away_cpi['observation_date']>='1960-01-01']

# rename 
food_home_cpi_clean = food_home_cpi_clean.rename(columns={'CUUR0000SAF11':'food_home_cpi'})
food_away_cpi_clean = food_away_cpi_clean.rename(columns={'CUUR0000SEFV':'food_away_cpi'})

# merge 
both_cpi = food_home_cpi_clean.merge(food_away_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_home = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_home_cpi"].values
base_away = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_away_cpi"].values

both_cpi['food_home_cpi_reindex'] = (both_cpi['food_home_cpi'] / base_home) * 100
both_cpi['food_away_cpi_reindex'] = (both_cpi['food_away_cpi'] / base_away) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [139]:
# line chart of food at home vs food away from home cpis 

# data 
x = both_cpi["observation_date"]
food_home_cpi = both_cpi["food_home_cpi_reindex"]
food_away_cpi = both_cpi["food_away_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Food at Home CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_away_cpi,
        mode="lines",
        name="Food Away from Home",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Food Away from Home CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food at Home vs. Away from Home Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False
    )
)

fig.show()

In [140]:
# save fig 
fig.write_html(f"../website/charts/home_vs_away.html", include_plotlyjs="cdn", full_html=False)

## Grocery Category Comparison 

Grocery categories cpi comparison = relative price trajectory of different food types, is cheaper food less healthy now


In [150]:
# import data 

cereal_bakery = pd.read_csv("../data/categories/CUUR0000SAF111.csv", low_memory=False)
meat_fish_eggs = pd.read_csv("../data/categories/CUUR0000SAF112.csv", low_memory=False)
fruit_veg = pd.read_csv("../data/categories/CUUR0000SAF113.csv", low_memory=False)
beverage = pd.read_csv("../data/categories/CUUR0000SAF114.csv", low_memory=False)

# clean data 

# filter to 1960 and later 
cereal_bakery = cereal_bakery.loc[cereal_bakery['observation_date']>='1967-01-01']
meat_fish_eggs = meat_fish_eggs.loc[meat_fish_eggs['observation_date']>='1967-01-01']
fruit_veg = fruit_veg.loc[fruit_veg['observation_date']>='1967-01-01']
beverage = beverage.loc[beverage['observation_date']>='1967-01-01']

# rename 
cereal_bakery = cereal_bakery.rename(columns={'CUUR0000SAF111':'cereal_bakery_cpi'})
meat_fish_eggs = meat_fish_eggs.rename(columns={'CUUR0000SAF112':'meat_fish_eggs_cpi'})
fruit_veg = fruit_veg.rename(columns={'CUUR0000SAF113':'fruit_veg_cpi'})
beverage = beverage.rename(columns={'CUUR0000SAF114':'beverage_cpi'})

# reindex to January 1967 = 100
base_cb = cereal_bakery.loc[cereal_bakery['observation_date']=="1967-01-01", "cereal_bakery_cpi"].values
base_mfe = meat_fish_eggs.loc[meat_fish_eggs['observation_date']=="1967-01-01", "meat_fish_eggs_cpi"].values
base_fv = fruit_veg.loc[fruit_veg['observation_date']=="1967-01-01", "fruit_veg_cpi"].values
base_b = beverage.loc[beverage['observation_date']=="1967-01-01", "beverage_cpi"].values

cereal_bakery['cereal_bakery_cpi_reindex'] = (cereal_bakery['cereal_bakery_cpi'] / base_cb) * 100
meat_fish_eggs['meat_fish_eggs_cpi_reindex'] = (meat_fish_eggs['meat_fish_eggs_cpi'] / base_mfe) * 100
fruit_veg['fruit_veg_cpi_reindex'] = (fruit_veg['fruit_veg_cpi'] / base_fv) * 100
beverage['beverage_cpi_reindex'] = (beverage['beverage_cpi'] / base_b) * 100

# drop na 
cereal_bakery = cereal_bakery.dropna()
meat_fish_eggs = meat_fish_eggs.dropna()
fruit_veg = fruit_veg.dropna()
beverage = beverage.dropna()

In [156]:
# line chart of grocery category cpis 

# data 
x = cereal_bakery["observation_date"]
cb = cereal_bakery["cereal_bakery_cpi_reindex"]
mfe = meat_fish_eggs["meat_fish_eggs_cpi_reindex"]
fv = fruit_veg["fruit_veg_cpi_reindex"]
b = beverage["beverage_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=cb,
        mode="lines",
        name="Cereal and Bakery",
        line=dict(
            color="#c48b22",
            width=4
        ),
        hovertemplate=
        "Cereal and Bakery CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=mfe,
        mode="lines",
        name="Meat, Poultry, Fish, Eggs",
        line=dict(
            color="#b84141",
            width=4
        ),
        hovertemplate=
        "Meat, Poultry, Fish, Eggs CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=fv,
        mode="lines",
        name="Fruit and Vegetables",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Fruit and Vegetables CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=b,
        mode="lines",
        name="Beverages",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Beverages CPI: %{y:.1f}<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Grocery Price Inflation by Category",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=13
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1
    )
)

fig.show()

In [157]:
# save fig 
fig.write_html(f"../website/charts/grocery_categories.html", include_plotlyjs="cdn", full_html=False)

## Animated Bar Chart by Outlet 

Monthly sales by outlet = compare sales by outlet over time in order to see whether more people are food at grocery stores, large stores like costco, restaurants, etc

In [24]:
# import data 

sales_by_outlet = pd.read_csv("../data/usda/monthly_sales_by_outlet.csv", low_memory=False)

In [ ]:
# clean data 

# select columns 
sales_by_outlet_clean = sales_by_outlet[['Year',
                                   'Month',
                                   'Food sales at grocery stores (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at warehouse club and supercenter (millions of constant U.S. dollars (1988=100))',
                                   'Other food-at-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at full-service restaurants (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at limited-service restaurant (millions of constant U.S. dollars (1988=100))',
                                   'Other food-away-from-home sales (millions of constant U.S. dollars (1988=100))']]

# rename columns 
sales_by_outlet_clean = sales_by_outlet_clean.rename(columns = {
    'Food sales at grocery stores (millions of constant U.S. dollars (1988=100))': 'grocery_stores',
    'Food sales at warehouse club and supercenter (millions of constant U.S. dollars (1988=100))': 'warehouse_supercenter',
    'Other food-at-home sales (millions of constant U.S. dollars (1988=100))': 'other_food_at_home',
    'Food sales at full-service restaurants (millions of constant U.S. dollars (1988=100))': 'full_restaurants',
    'Food sales at limited-service restaurant (millions of constant U.S. dollars (1988=100))': 'limited_restaurants',
    'Other food-away-from-home sales (millions of constant U.S. dollars (1988=100))': 'other_food_away',
    # 'Total food-at-home sales (millions of constant U.S. dollars (1988=100))': 'total_food_at_home',
    # 'Total food-away-from-home sales (millions of constant U.S. dollars (1988=100))': 'total_food_away',
    # 'Total food sales (millions of constant U.S. dollars (1988=100))': 'total_overall'
})

# combine year and month into date column 
sales_by_outlet_clean['date'] = pd.to_datetime(sales_by_outlet_clean['Month'] + ' ' + sales_by_outlet_clean['Year'].astype(str), format='%B %Y')
sales_by_outlet_clean['month_year'] = sales_by_outlet_clean['date'].dt.strftime('%B %Y')

# convert to long format
sales_by_outlet_long = sales_by_outlet_clean.melt(
    id_vars=['Year', 'Month', 'date', 'month_year'],
    value_vars=[
        'grocery_stores',
        'warehouse_supercenter',
        'other_food_at_home',
        'full_restaurants',
        'limited_restaurants',
        'other_food_away',
        # 'total_food_at_home',
        # 'total_food_away',
        # 'total_overall'
    ],
    var_name='category',
    value_name='sales'
)

# convert sales to float 
sales_by_outlet_long['sales'] = sales_by_outlet_long['sales'].astype(str).str.replace(',', '', regex=False).astype(float)

# sort values 
sales_by_outlet_long = sales_by_outlet_long.sort_values(by=['date','category'])

In [ ]:
# make plot 

# colors
color_map = {
    'grocery_stores': "#458dbd",
    'warehouse_supercenter': "#67a384",
    'other_food_at_home': "#adddfc",
    'full_restaurants': "#b84141",
    'limited_restaurants': "#e38a72",
    'other_food_away': "#e3b96b",
}

# plot 
fig = px.bar(sales_by_outlet_long, 
             x="category", 
             y="sales", 
             color="category",
             color_discrete_map=color_map,
             animation_frame="date",
             animation_group="category",
             range_y=[0,25000],
             custom_data=['month_year'],
             )

fig.update_traces(
    marker_line_width=0,
)

fig.for_each_trace(
    lambda t: t.update(
        hovertemplate=
            '<b>%{x}</b><br>' +
            'Date: %{customdata}<br>' +
            'Sales: $%{y:,.0f}<br>' +
            '<extra></extra>',
        marker_line_width=0
    )
)

fig.update_layout(
    # title
    title={
        'text': 'Food Sales by Outlet Type over Time',
        'x': 0.02,
        'xanchor': 'left',
        'font': {
            'size': 24,
            'family': 'Playfair Display',
            'color': '#f5f5f5'
        }
    },

    # plot background
    plot_bgcolor='#1a1a1a',

    # entire figure background
    paper_bgcolor='#1a1a1a',

    # x-axis
    xaxis=dict(
        title='Outlet Category',
        title_font=dict(
            size=17,
            family="Source Sans 3",
            color='#f5f5f5'
        ),
        tickfont=dict(
            size=14,
            color='#f5f5f5'
        )
    ),

    # y-axis
    yaxis=dict(
        title='Millions of constant U.S. dollars<br>(1988=100)',
        title_font=dict(
            size=17,
            family="Source Sans 3",
            color='#f5f5f5'
        ),
        tickfont=dict(
            size=16,
            color='#f5f5f5'
        )
    ),

    # legend
    showlegend=False
)

fig.update_xaxes(
    tickvals=[
        'grocery_stores',
        'warehouse_supercenter',
        'other_food_at_home',
        'full_restaurants',
        'limited_restaurants',
        'other_food_away'
    ],
    ticktext=[
        'Grocery Stores',
        'Warehouse Clubs',
        'Other Food at Home',
        'Full Restaurants',
        'Limited Restaurants',
        'Other Food Away from Home'
    ]
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zerolinecolor="rgba(255,255,255,0.08)",
)

for step in fig.layout.sliders[0].steps:
    step["label"] = step["label"][:7]

fig.update_layout(

    # animation slider
    sliders=[{
        'currentvalue': {
            'font': {'color': '#f5f5f5'},
            'prefix': 'Date: ',
            'visible': True,
        },

        # tick labels on slider
        'tickcolor': "rgba(245, 245, 245, 0.2)",

        # slider styling
        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",
        'borderwidth': 1,

        # font for slider step labels
        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        }
    }],

    # buttons
    updatemenus=[{
        'type': 'buttons',

        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        },

        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",

    }]
)

fig.show()

In [72]:
# save fig 
fig.write_html(f"../website/charts/spending_by_outlet.html", include_plotlyjs="cdn", full_html=False)